# Holdout 70:30 Error Analysis

`train.csv` を 7:3（Stratified）で分割し、検証データの誤分類を分析します。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

import lightgbm as lgb

SEED = 42
VALID_SIZE = 0.3
DATA_PATH = "data/train.csv"

In [ ]:
train = pd.read_csv(DATA_PATH)
print(train.shape)
train.head()

In [ ]:
# 誤分類サンプルを抽出して確認
va_result = train_fe.loc[idx_va, ["id", "class"]].copy()
va_result["y_true"] = target_encoder.inverse_transform(y_va)
va_result["y_pred"] = target_encoder.inverse_transform(pred_va)
va_result["pred_conf"] = proba_va.max(axis=1)
va_result["is_error"] = va_result["y_true"] != va_result["y_pred"]

errors = va_result[va_result["is_error"]].copy()
print("n_valid:", len(va_result), "n_error:", len(errors), "error_rate:", len(errors) / len(va_result))

# どの取り違えが多いか
error_pairs = (
    errors.groupby(["y_true", "y_pred"]).size().sort_values(ascending=False).rename("count").reset_index()
)
error_pairs.head(10)

In [ ]:
# 高確信なのに間違えているケースを確認
high_conf_errors = errors.sort_values("pred_conf", ascending=False)
high_conf_errors.head(20)

In [ ]:
# 誤分類サンプルに特徴量を戻して可視化・解析に使う
error_detail = train_fe.loc[errors.index, ["id", "class", "redshift", "u", "g", "r", "i", "z", "u_g", "g_r", "r_i", "i_z", "u_r", "g_i", "u_z", "redshift_log1p", "spectral_type", "galaxy_population"]].copy()
error_detail["y_pred"] = errors["y_pred"].values
error_detail["pred_conf"] = errors["pred_conf"].values

error_detail.head(20)

## 追加対策: クラス別バイアス補正

`argmax` そのままだとクラス間の境界が均等で、`Balanced Accuracy` を最適化しきれないことがあります。  
そこで holdout 上でクラスごとの `log(prob)` に定数バイアスを足し、`Balanced Accuracy` が最大になる補正を探索します。

In [ ]:
# バイアス探索（3クラス）
# pred = argmax(log(proba) + bias)
log_proba_va = np.log(np.clip(proba_va, 1e-15, 1.0))

grid = np.arange(-0.06, 0.061, 0.01)
best_bias = np.zeros(len(target_encoder.classes_))
best_score_bias = score

for b0 in grid:
    for b1 in grid:
        for b2 in grid:
            bias = np.array([b0, b1, b2])
            pred_tmp = (log_proba_va + bias).argmax(axis=1)
            s = balanced_accuracy_score(y_va, pred_tmp)
            if s > best_score_bias:
                best_score_bias = s
                best_bias = bias.copy()

pred_va_bias = (log_proba_va + best_bias).argmax(axis=1)

print("base balanced accuracy:", f"{score:.5f}")
print("bias-corrected balanced accuracy:", f"{best_score_bias:.5f}")
print("best bias (class order):", list(target_encoder.classes_), best_bias.tolist())

In [ ]:
# 改善前後の誤分類件数と混同行列を比較
n_err_base = int((pred_va != y_va).sum())
n_err_bias = int((pred_va_bias != y_va).sum())

print("n_valid:", len(y_va))
print("base error count:", n_err_base, "error rate:", n_err_base / len(y_va))
print("bias-corrected error count:", n_err_bias, "error rate:", n_err_bias / len(y_va))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_base = confusion_matrix(y_va, pred_va, normalize="true")
cm_bias = confusion_matrix(y_va, pred_va_bias, normalize="true")

ConfusionMatrixDisplay(cm_base, display_labels=target_encoder.classes_).plot(ax=axes[0], cmap="Blues", values_format=".3f", colorbar=False)
axes[0].set_title("Base")

ConfusionMatrixDisplay(cm_bias, display_labels=target_encoder.classes_).plot(ax=axes[1], cmap="Blues", values_format=".3f", colorbar=False)
axes[1].set_title("Bias-corrected")

plt.tight_layout()
plt.show()

In [ ]:
# バイアス補正後の誤分類ペアを確認
va_result_bias = train_fe.loc[idx_va, ["id", "class"]].copy()
va_result_bias["y_true"] = target_encoder.inverse_transform(y_va)
va_result_bias["y_pred"] = target_encoder.inverse_transform(pred_va_bias)
va_result_bias["pred_conf"] = proba_va.max(axis=1)
va_result_bias["is_error"] = va_result_bias["y_true"] != va_result_bias["y_pred"]

errors_bias = va_result_bias[va_result_bias["is_error"]].copy()

error_pairs_bias = (
    errors_bias.groupby(["y_true", "y_pred"]).size().sort_values(ascending=False).rename("count").reset_index()
)

error_pairs_bias.head(10)